# WoWAH Player Lifecycle Segmentation

Bu notebook, full `fct_player_daily` katmanindan aylik snapshot bazli lifecycle feature ve segment martlarini uretir.


## Lifecycle Segmentation Nedir?

Gaming analytics tarafinda lifecycle segmentation, oyunculari tek bir statik label yerine zaman icinde degisen davranis kaliplarina gore gruplamaktir. Burada segmentation aylik snapshot mantigiyla kurulur; boylece her ay sonunda hangi segmentin buyudugu, daraldigi veya risk olusturdugu gorulur.


## Proxy ve Engagement Mantigi

- **Player proxy**: Bu dataset stabil oyuncu kimligi olarak `avatar_id` sagladigi icin tum feature ve segment hesaplarinda onu kullaniyoruz.
- **Engagement proxy'leri**: Gelir veya CLV olmadigi icin deger sinyallerini `active_days`, `observations`, `level_gain`, `zone diversity` ve `guild participation` uzerinden kuruyoruz.
- **Mutually exclusive segments**: Her avatar her snapshot'ta yalnizca tek segmente gider; bu sayede dashboard ve LiveOps aksiyonlari cakismaz.


## Segment Tanimlari ve LiveOps Aksiyonlari

1. **Lapsed Players**: `recency_days > 30` -> win-back campaign / return incentive
2. **Reactivated Players**: son 7 gunde donus ve onceki gap > 30 gun -> return reinforcement
3. **New Explorers**: `days_since_first_seen <= 7` -> onboarding support
4. **At-Risk Players**: 8-30 gun recency ve aktivite dususu -> targeted reactivation challenge
5. **Core Engaged**: yuksek 30 gun aktivitesi -> advanced content / high-engagement event
6. **Fast Progressors**: hizli seviye ilerlemesi -> progression-focused event
7. **Social/Guild Engaged**: guild davranisi guclu -> guild-based event / group challenge
8. **Casual Returners**: hafif ama son 30 gunde aktif -> weekend challenge
9. **Low Activity / Other**: diger tum oyuncular -> general engagement monitoring


## Data Quality Limitations

- Monetization sinyali olmadigi icin revenue veya CLV kullanilmiyor.
- `guild_member_flag_latest` tum tarih boyunca her satirda dolu degil; guild sinyalleri mevcut gozlemler kadar guvenilir.
- `level_gain_day` icinde negatif satirlar mevcut; bu satirlari silmiyoruz, warning olarak tasiyoruz.
- Segmentler behavior proxy'lerinden uretildigi icin oyuncu niyeti dogrudan gozlemlenmiyor.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

print('DuckDB ve pandas hazir.')


In [ ]:
project_root = Path.cwd()
processed_dir = project_root / 'data/processed'
outputs_dir = project_root / 'data/outputs'
sql_file = project_root / 'sql/04_player_lifecycle_segmentation.sql'

fct_source_path = processed_dir / 'fct_player_daily.parquet'
agg_daily_metrics_path = processed_dir / 'agg_daily_metrics.parquet'
cohort_retention_path = processed_dir / 'agg_cohort_retention.parquet'

if not fct_source_path.exists():
    raise FileNotFoundError(f'fct_player_daily parquet bulunamadi: {fct_source_path}')
if not agg_daily_metrics_path.exists():
    raise FileNotFoundError(f'agg_daily_metrics parquet bulunamadi: {agg_daily_metrics_path}')

features_output_path = processed_dir / 'mart_player_lifecycle_features.parquet'
segments_output_path = processed_dir / 'mart_player_segments.parquet'
segment_monthly_output_path = processed_dir / 'agg_segment_monthly.parquet'
validation_output_path = outputs_dir / 'segmentation_validation_summary.csv'

print('Primary input source:', fct_source_path)
print('Daily metrics source:', agg_daily_metrics_path)
print('Cohort retention source exists:', cohort_retention_path.exists())
print('SQL file:', sql_file)
print('Features output:', features_output_path)
print('Segments output:', segments_output_path)
print('Segment monthly output:', segment_monthly_output_path)
print('Validation output:', validation_output_path)


In [ ]:
con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE VIEW fct_player_daily AS SELECT * FROM read_parquet('{fct_source_path}')"
)
con.execute(
    f"CREATE OR REPLACE VIEW agg_daily_metrics AS SELECT * FROM read_parquet('{agg_daily_metrics_path}')"
)
if cohort_retention_path.exists():
    con.execute(
        f"CREATE OR REPLACE VIEW agg_cohort_retention AS SELECT * FROM read_parquet('{cohort_retention_path}')"
    )

con.execute(
    '''
    CREATE OR REPLACE VIEW segmentation_source_validation AS
    SELECT
        COUNT(*) AS fct_row_count,
        COUNT(DISTINCT avatar_id) AS unique_avatars,
        MIN(activity_date) AS min_activity_date,
        MAX(activity_date) AS max_activity_date,
        COUNT(*) - COUNT(DISTINCT (CAST(activity_date AS VARCHAR) || '|' || CAST(avatar_id AS VARCHAR))) AS duplicate_avatar_day_rows,
        SUM(CASE WHEN level_gain_day < 0 THEN 1 ELSE 0 END) AS negative_level_gain_day_count
    FROM fct_player_daily
    '''
)

source_validation_overview = con.execute('SELECT * FROM segmentation_source_validation').df()
source_validation_overview


In [ ]:
con.execute(sql_file.read_text())

model_overview = con.execute(
    '''
    SELECT
        (SELECT COUNT(*) FROM lifecycle_monthly_snapshots) AS total_snapshots,
        (SELECT MIN(snapshot_date) FROM lifecycle_monthly_snapshots) AS min_snapshot_date,
        (SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots) AS max_snapshot_date,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features) AS feature_rows,
        (SELECT COUNT(*) FROM mart_player_segments) AS segment_rows,
        (SELECT COUNT(*) FROM agg_segment_monthly) AS agg_segment_monthly_rows
    '''
).df()

print('Lifecycle segmentation views SQL file uzerinden olusturuldu.')
model_overview


In [ ]:
latest_snapshot_date = con.execute(
    'SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots'
).fetchone()[0]

latest_segment_distribution = con.execute(
    '''
    SELECT
        snapshot_date,
        lifecycle_segment,
        segment_size,
        segment_share,
        avg_recency_days,
        avg_active_days_30,
        avg_observations_30,
        avg_level_current,
        avg_level_gain_30
    FROM agg_segment_monthly
    WHERE snapshot_date = ?
    ORDER BY segment_size DESC, lifecycle_segment ASC
    '''
    , [latest_snapshot_date]
).df()

feature_preview = con.execute(
    '''
    SELECT *
    FROM mart_player_lifecycle_features
    WHERE snapshot_date = ?
    ORDER BY avatar_id ASC
    LIMIT 10
    '''
    , [latest_snapshot_date]
).df()

print('Latest snapshot date:', latest_snapshot_date)
print()
print('Latest segment distribution')
print(latest_segment_distribution.to_string(index=False))
print()
print('Lifecycle feature preview')
print(feature_preview.to_string(index=False))


In [ ]:
con.execute(
    '''
    CREATE OR REPLACE VIEW segmentation_validation_overview AS
    SELECT
        (SELECT COUNT(*) FROM lifecycle_monthly_snapshots) AS total_snapshots,
        (SELECT MIN(snapshot_date) FROM lifecycle_monthly_snapshots) AS min_snapshot_date,
        (SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots) AS max_snapshot_date,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features) AS total_feature_rows,
        (SELECT COUNT(*) FROM mart_player_segments) AS total_segment_rows,
        (SELECT unique_avatars FROM segmentation_source_validation) AS unique_avatars_overall,
        (SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots) AS latest_snapshot_date,
        (SELECT COUNT(*) FROM mart_player_segments WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots)) AS latest_snapshot_total_avatars,
        (SELECT COUNT(*) FROM mart_player_segments WHERE lifecycle_segment IS NULL) AS null_lifecycle_segment_rows,
        (SELECT COUNT(*) - COUNT(DISTINCT (CAST(snapshot_date AS VARCHAR) || '|' || CAST(avatar_id AS VARCHAR))) FROM mart_player_segments) AS duplicate_avatar_snapshot_rows,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features WHERE recency_days < 0) AS negative_recency_rows,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features WHERE active_days_30 > 30) AS active_days_30_gt_30_rows,
        (
            SELECT COUNT(*)
            FROM (
                SELECT snapshot_date, ABS(SUM(segment_share) - 1.0) AS share_deviation
                FROM agg_segment_monthly
                GROUP BY 1
            ) x
            WHERE share_deviation > 0.0001
        ) AS segment_share_sum_not_close_to_one_snapshots
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW segmentation_sanity_checks AS
    SELECT
        (SELECT COUNT(*) FROM mart_player_lifecycle_features WHERE active_days_7 < 0 OR active_days_7 > 7) AS active_days_7_out_of_range_rows,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features WHERE active_days_30 < 0 OR active_days_30 > 30) AS active_days_30_out_of_range_rows,
        (SELECT COUNT(*) FROM mart_player_lifecycle_features WHERE active_days_prev_30 < 0 OR active_days_prev_30 > 30) AS active_days_prev_30_out_of_range_rows,
        (
            SELECT COUNT(*)
            FROM mart_player_segments s
            WHERE s.lifecycle_segment = 'Lapsed Players'
              AND s.recency_days <= 30
        ) AS invalid_lapsed_rows,
        (
            SELECT COUNT(*)
            FROM mart_player_segments s
            JOIN mart_player_lifecycle_features f
                USING (snapshot_date, avatar_id)
            WHERE s.lifecycle_segment = 'New Explorers'
              AND f.days_since_first_seen > 7
        ) AS invalid_new_explorer_rows,
        (
            SELECT negative_level_gain_day_count
            FROM segmentation_source_validation
        ) AS negative_input_level_gain_day_count
    '''
)

validation_overview = con.execute('SELECT * FROM segmentation_validation_overview').df()
sanity_checks = con.execute('SELECT * FROM segmentation_sanity_checks').df()

print('Validation overview')
print(validation_overview.to_string(index=False))
print()
print('Sanity checks')
print(sanity_checks.to_string(index=False))


In [ ]:
con.execute(
    '''
    CREATE OR REPLACE VIEW latest_snapshot_segment_distribution AS
    SELECT
        snapshot_date,
        lifecycle_segment,
        segment_size,
        segment_share,
        avg_recency_days,
        avg_active_days_30,
        avg_observations_30,
        avg_level_current,
        avg_level_gain_30,
        guild_member_share,
        fast_progressor_share,
        reactivated_share
    FROM agg_segment_monthly
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM lifecycle_monthly_snapshots)
    ORDER BY segment_size DESC, lifecycle_segment ASC
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW top_10_fastest_growing_segments_latest_3m AS
    SELECT *
    FROM lifecycle_segment_growth_latest_3m
    ORDER BY segment_size_change_vs_3m_ago DESC NULLS LAST, lifecycle_segment ASC
    LIMIT 10
    '''
)

con.execute(
    '''
    CREATE OR REPLACE VIEW top_10_shrinking_segments_latest_3m AS
    SELECT *
    FROM lifecycle_segment_growth_latest_3m
    ORDER BY segment_size_change_vs_3m_ago ASC NULLS LAST, lifecycle_segment ASC
    LIMIT 10
    '''
)

latest_snapshot_segment_distribution = con.execute(
    'SELECT * FROM latest_snapshot_segment_distribution'
).df()
fastest_growing_segments = con.execute(
    'SELECT * FROM top_10_fastest_growing_segments_latest_3m'
).df()
shrinking_segments = con.execute(
    'SELECT * FROM top_10_shrinking_segments_latest_3m'
).df()

print('Latest snapshot segment distribution')
print(latest_snapshot_segment_distribution.to_string(index=False))
print()
print('Fastest growing segments over latest 3 months')
print(fastest_growing_segments.to_string(index=False))
print()
print('Shrinking segments over latest 3 months')
print(shrinking_segments.to_string(index=False))


In [ ]:
features_output_tmp = features_output_path.with_suffix(features_output_path.suffix + '.tmp')
segments_output_tmp = segments_output_path.with_suffix(segments_output_path.suffix + '.tmp')
segment_monthly_output_tmp = segment_monthly_output_path.with_suffix(segment_monthly_output_path.suffix + '.tmp')
validation_output_tmp = validation_output_path.with_suffix(validation_output_path.suffix + '.tmp')

for temp_path in [features_output_tmp, segments_output_tmp, segment_monthly_output_tmp, validation_output_tmp]:
    if temp_path.exists():
        temp_path.unlink()

con.execute(
    f"COPY (SELECT * FROM mart_player_lifecycle_features) TO '{features_output_tmp}' (FORMAT PARQUET)"
)
con.execute(
    f"COPY (SELECT * FROM mart_player_segments) TO '{segments_output_tmp}' (FORMAT PARQUET)"
)
con.execute(
    f"COPY (SELECT * FROM agg_segment_monthly) TO '{segment_monthly_output_tmp}' (FORMAT PARQUET)"
)

con.execute(
    '''
    CREATE OR REPLACE VIEW segmentation_validation_summary AS
    WITH summary_rows AS (
        SELECT
            'summary' AS section,
            1 AS position,
            'total_snapshots' AS metric_name,
            CAST(total_snapshots AS VARCHAR) AS metric_value,
            NULL::DATE AS snapshot_date,
            NULL::VARCHAR AS lifecycle_segment,
            NULL::BIGINT AS segment_size,
            NULL::DOUBLE AS segment_share,
            NULL::DOUBLE AS avg_recency_days,
            NULL::DOUBLE AS avg_active_days_30,
            NULL::DOUBLE AS avg_observations_30,
            NULL::DOUBLE AS avg_level_current,
            NULL::DOUBLE AS avg_level_gain_30,
            NULL::DOUBLE AS guild_member_share,
            NULL::DOUBLE AS fast_progressor_share,
            NULL::DOUBLE AS reactivated_share,
            NULL::BIGINT AS segment_size_change_vs_3m_ago,
            NULL::DOUBLE AS segment_share_change_vs_3m_ago
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 2, 'min_snapshot_date', CAST(min_snapshot_date AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 3, 'max_snapshot_date', CAST(max_snapshot_date AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 4, 'total_rows_in_mart_player_lifecycle_features', CAST(total_feature_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 5, 'total_rows_in_mart_player_segments', CAST(total_segment_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 6, 'unique_avatars_overall', CAST(unique_avatars_overall AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 7, 'latest_snapshot_date', CAST(latest_snapshot_date AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 8, 'latest_snapshot_total_avatars', CAST(latest_snapshot_total_avatars AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 9, 'null_lifecycle_segment_rows', CAST(null_lifecycle_segment_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 10, 'duplicate_avatar_id_snapshot_date_rows', CAST(duplicate_avatar_snapshot_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 11, 'recency_days_lt_zero_rows', CAST(negative_recency_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 12, 'active_days_30_gt_30_rows', CAST(active_days_30_gt_30_rows AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
        UNION ALL
        SELECT 'summary', 13, 'segment_share_not_close_to_one_snapshots', CAST(segment_share_sum_not_close_to_one_snapshots AS VARCHAR), NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
        FROM segmentation_validation_overview
    ),
    latest_distribution_rows AS (
        SELECT
            'latest_snapshot_segment_distribution' AS section,
            ROW_NUMBER() OVER (ORDER BY segment_size DESC, lifecycle_segment ASC) AS position,
            'latest_snapshot_segment' AS metric_name,
            CAST(segment_size AS VARCHAR) AS metric_value,
            snapshot_date,
            lifecycle_segment,
            segment_size,
            segment_share,
            avg_recency_days,
            avg_active_days_30,
            avg_observations_30,
            avg_level_current,
            avg_level_gain_30,
            guild_member_share,
            fast_progressor_share,
            reactivated_share,
            NULL::BIGINT AS segment_size_change_vs_3m_ago,
            NULL::DOUBLE AS segment_share_change_vs_3m_ago
        FROM latest_snapshot_segment_distribution
    ),
    fastest_growth_rows AS (
        SELECT
            'top_10_fastest_growing_segments_latest_3m' AS section,
            ROW_NUMBER() OVER (ORDER BY segment_size_change_vs_3m_ago DESC NULLS LAST, lifecycle_segment ASC) AS position,
            'segment_growth' AS metric_name,
            CAST(segment_size_change_vs_3m_ago AS VARCHAR) AS metric_value,
            snapshot_date,
            lifecycle_segment,
            segment_size,
            segment_share,
            NULL::DOUBLE AS avg_recency_days,
            NULL::DOUBLE AS avg_active_days_30,
            NULL::DOUBLE AS avg_observations_30,
            NULL::DOUBLE AS avg_level_current,
            NULL::DOUBLE AS avg_level_gain_30,
            NULL::DOUBLE AS guild_member_share,
            NULL::DOUBLE AS fast_progressor_share,
            NULL::DOUBLE AS reactivated_share,
            segment_size_change_vs_3m_ago,
            segment_share_change_vs_3m_ago
        FROM top_10_fastest_growing_segments_latest_3m
    ),
    shrinking_rows AS (
        SELECT
            'top_10_shrinking_segments_latest_3m' AS section,
            ROW_NUMBER() OVER (ORDER BY segment_size_change_vs_3m_ago ASC NULLS LAST, lifecycle_segment ASC) AS position,
            'segment_growth' AS metric_name,
            CAST(segment_size_change_vs_3m_ago AS VARCHAR) AS metric_value,
            snapshot_date,
            lifecycle_segment,
            segment_size,
            segment_share,
            NULL::DOUBLE AS avg_recency_days,
            NULL::DOUBLE AS avg_active_days_30,
            NULL::DOUBLE AS avg_observations_30,
            NULL::DOUBLE AS avg_level_current,
            NULL::DOUBLE AS avg_level_gain_30,
            NULL::DOUBLE AS guild_member_share,
            NULL::DOUBLE AS fast_progressor_share,
            NULL::DOUBLE AS reactivated_share,
            segment_size_change_vs_3m_ago,
            segment_share_change_vs_3m_ago
        FROM top_10_shrinking_segments_latest_3m
    )
    SELECT * FROM summary_rows
    UNION ALL
    SELECT * FROM latest_distribution_rows
    UNION ALL
    SELECT * FROM fastest_growth_rows
    UNION ALL
    SELECT * FROM shrinking_rows
    '''
)

con.execute(
    f"COPY segmentation_validation_summary TO '{validation_output_tmp}' (FORMAT CSV, HEADER)"
)

features_output_tmp.replace(features_output_path)
segments_output_tmp.replace(segments_output_path)
segment_monthly_output_tmp.replace(segment_monthly_output_path)
validation_output_tmp.replace(validation_output_path)

print('Outputs written.')




In [ ]:
sanity_row = sanity_checks.iloc[0].to_dict()
validation_row = validation_overview.iloc[0].to_dict()

warnings = []
warning_map = {
    'negative_input_level_gain_day_count': 'negative_level_gain_day_present_in_input',
    'active_days_7_out_of_range_rows': 'active_days_7_out_of_range_rows_present',
    'active_days_30_out_of_range_rows': 'active_days_30_out_of_range_rows_present',
    'active_days_prev_30_out_of_range_rows': 'active_days_prev_30_out_of_range_rows_present',
    'invalid_lapsed_rows': 'invalid_lapsed_segment_rows_present',
    'invalid_new_explorer_rows': 'invalid_new_explorer_segment_rows_present',
}
for key, warning_text in warning_map.items():
    if int(sanity_row.get(key, 0) or 0) > 0:
        warnings.append(warning_text)
if int(validation_row.get('segment_share_sum_not_close_to_one_snapshots', 0) or 0) > 0:
    warnings.append('segment_share_sum_not_close_to_one_snapshots_present')
if int(validation_row.get('null_lifecycle_segment_rows', 0) or 0) > 0:
    warnings.append('null_lifecycle_segment_rows_present')
if int(validation_row.get('duplicate_avatar_snapshot_rows', 0) or 0) > 0:
    warnings.append('duplicate_avatar_snapshot_rows_present')
if int(validation_row.get('negative_recency_rows', 0) or 0) > 0:
    warnings.append('negative_recency_rows_present')
if int(validation_row.get('active_days_30_gt_30_rows', 0) or 0) > 0:
    warnings.append('active_days_30_gt_30_rows_present')

warning_summary = 'none' if not warnings else '; '.join(warnings)
latest_distribution_text = '; '.join(
    f"{row['lifecycle_segment']}={int(row['segment_size'])} ({row['segment_share']:.4f})"
    for _, row in latest_snapshot_segment_distribution.iterrows()
)
largest_segment_row = latest_snapshot_segment_distribution.iloc[0]
largest_segment_text = f"{largest_segment_row['lifecycle_segment']} ({int(largest_segment_row['segment_size'])})"
fastest_growing_row = fastest_growing_segments.iloc[0]
fastest_growing_text = (
    f"{fastest_growing_row['lifecycle_segment']} "
    f"({int(fastest_growing_row['segment_size_change_vs_3m_ago']) if pd.notna(fastest_growing_row['segment_size_change_vs_3m_ago']) else 'NA'})"
)
files_created = '; '.join([
    str(features_output_path),
    str(segments_output_path),
    str(segment_monthly_output_path),
    str(validation_output_path),
])

final_report = con.execute(
    '''
    SELECT 'input_source_used' AS metric_name, ? AS metric_value
    UNION ALL
    SELECT 'total_snapshots', CAST((SELECT total_snapshots FROM segmentation_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'snapshot_date_range', CAST((SELECT min_snapshot_date FROM segmentation_validation_overview) AS VARCHAR) || ' -> ' || CAST((SELECT max_snapshot_date FROM segmentation_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'total_rows_in_mart_player_segments', CAST((SELECT total_segment_rows FROM segmentation_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'unique_avatars', CAST((SELECT unique_avatars_overall FROM segmentation_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'latest_snapshot_date', CAST((SELECT latest_snapshot_date FROM segmentation_validation_overview) AS VARCHAR)
    UNION ALL
    SELECT 'latest_segment_distribution', ?
    UNION ALL
    SELECT 'largest_segment', ?
    UNION ALL
    SELECT 'fastest_growing_segment_over_latest_3_months', ?
    UNION ALL
    SELECT 'data_quality_warnings', ?
    UNION ALL
    SELECT 'files_created', ?
    '''
    , [str(fct_source_path), latest_distribution_text, largest_segment_text, fastest_growing_text, warning_summary, files_created]
).df()

final_report
